# Job Match ETL Pipeline

This notebook walks through the full pipeline in self-contained sections you can run one at a time.

**Pipeline overview:**
1. Install dependencies
2. Configuration (MongoDB + search settings)
3. MongoDB connection & helper functions
4. Indeed scraper
5. Run: scrape → store
6. Verify: preview what was stored

> **NLP sections** (resume parsing + cosine matching) will be added in the next notebook cells once this section is working.

---

## Section 1 — Install Dependencies

Run this cell once. After it finishes, you do **not** need to run it again unless you start a fresh environment.

- `playwright` — headless browser for scraping JavaScript-rendered pages
- `pymongo` — Python driver for MongoDB Atlas
- `python-dotenv` — loads your secrets from a `.env` file so they are never hardcoded

In [ ]:
import sys

# Install Python packages
%pip install playwright pymongo python-dotenv --quiet

# Download the headless Chromium browser that Playwright uses
# This is a ~150 MB one-time download
!playwright install chromium

print("\nAll dependencies installed.")

---
## Section 2 — Configuration

**Before running this cell**, create a file named `.env` in the same folder as this notebook.
Copy the block below into it and fill in your values:

```
MONGO_URI=mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority
MONGO_DB_NAME=job_pipeline
MONGO_COLLECTION_NAME=job_postings
SEARCH_QUERY=data analyst
SEARCH_LOCATION=Salt Lake City, UT
MAX_JOBS=20
```

Your MongoDB URI comes from: **Atlas Dashboard → your cluster → Connect → Drivers → copy the connection string**

> If you prefer not to use a `.env` file, you can directly edit the `CONFIG` dictionary in the cell below instead.

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into os.environ
load_dotenv(override=True)

# All settings live here. Values come from .env if set,
# or fall back to the defaults on the right side of `or`.
CONFIG = {
    # MongoDB
    "MONGO_URI":             os.getenv("MONGO_URI"),
    "MONGO_DB_NAME":         os.getenv("MONGO_DB_NAME",         "job_pipeline"),
    "MONGO_COLLECTION_NAME": os.getenv("MONGO_COLLECTION_NAME", "job_postings"),

    # Scraper
    "SEARCH_QUERY":          os.getenv("SEARCH_QUERY",          "data analyst"),
    "SEARCH_LOCATION":       os.getenv("SEARCH_LOCATION",       "Salt Lake City, UT"),
    "MAX_JOBS":              int(os.getenv("MAX_JOBS",           "20")),
}

# Confirm everything loaded (URI is partially masked for safety)
uri = CONFIG["MONGO_URI"] or "NOT SET"
masked_uri = uri[:30] + "..." if len(uri) > 30 else uri

print("Configuration loaded:")
print(f"  MONGO_URI             : {masked_uri}")
print(f"  MONGO_DB_NAME         : {CONFIG['MONGO_DB_NAME']}")
print(f"  MONGO_COLLECTION_NAME : {CONFIG['MONGO_COLLECTION_NAME']}")
print(f"  SEARCH_QUERY          : {CONFIG['SEARCH_QUERY']}")
print(f"  SEARCH_LOCATION       : {CONFIG['SEARCH_LOCATION']}")
print(f"  MAX_JOBS              : {CONFIG['MAX_JOBS']}")

if not CONFIG["MONGO_URI"]:
    print("\nWARNING: MONGO_URI is not set. Create a .env file or set it above.")

---
## Section 3 — MongoDB Helpers

This cell defines all database functions. Run it once to load them into memory —
it does **not** connect to Atlas yet, it just defines the functions.

| Function | What it does |
|---|---|
| `get_db()` | Opens a connection to Atlas and pings it to confirm it works |
| `get_collection(db)` | Returns the jobs collection; creates a unique index on `job_key` to prevent duplicates |
| `insert_jobs(collection, jobs)` | Inserts a list of job dicts, silently skipping any duplicate `job_key` |
| `get_all_jobs(collection)` | Returns every stored job as a list of plain dicts |
| `get_unscored_jobs(collection)` | Returns only jobs that have not been matched yet (used in the NLP step) |
| `update_match_score(collection, job_key, score)` | Writes the NLP cosine score back onto a job document |
| `get_top_matches(collection, n)` | Returns top-n jobs ranked by match score (used in reporting) |

In [ ]:
from datetime import datetime, timezone
from pymongo import MongoClient, ASCENDING
from pymongo.errors import DuplicateKeyError, ConnectionFailure


def get_db():
    """
    Connect to MongoDB Atlas and return the database object.
    Raises a clear error if the URI is missing or the connection fails.
    """
    uri = CONFIG["MONGO_URI"]
    if not uri:
        raise ValueError(
            "MONGO_URI is not set. Add it to your .env file or set it directly in CONFIG."
        )

    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=5000)
        # Ping Atlas — if this line passes, the connection is working
        client.admin.command("ping")
        print("Connected to MongoDB Atlas")
    except ConnectionFailure as e:
        raise ConnectionError(f"Could not connect to MongoDB Atlas: {e}")

    return client[CONFIG["MONGO_DB_NAME"]]


def get_collection(db):
    """
    Return the jobs collection and ensure a unique index on job_key exists.
    The unique index means inserting the same job twice will silently fail
    rather than creating a duplicate — safe to re-run every week.
    """
    col = db[CONFIG["MONGO_COLLECTION_NAME"]]
    col.create_index([("job_key", ASCENDING)], unique=True)
    return col


def insert_jobs(collection, jobs):
    """
    Insert a list of job dicts into MongoDB.
    Skips any job whose job_key is already in the collection.
    Returns a dict with inserted and skipped counts.
    """
    inserted = 0
    skipped  = 0

    for job in jobs:
        job["stored_at"] = datetime.now(timezone.utc)  # timestamp every document
        try:
            collection.insert_one(job)
            inserted += 1
            print(f"  Inserted : {job.get('title','?')} @ {job.get('company','?')}")
        except DuplicateKeyError:
            skipped += 1
            print(f"  Skipped  : {job.get('title','?')} (already in DB)")

    return {"inserted": inserted, "skipped": skipped}


def get_all_jobs(collection):
    """Return all job documents as plain Python dicts (no internal Mongo _id field)."""
    return list(collection.find({}, {"_id": 0}))


def get_unscored_jobs(collection):
    """Return jobs where match_score has not been set yet (used in the NLP step)."""
    return list(collection.find(
        {"$or": [{"match_score": None}, {"match_score": {"$exists": False}}]},
        {"_id": 0},
    ))


def update_match_score(collection, job_key, score):
    """Write the NLP cosine similarity score back onto an existing job document."""
    collection.update_one(
        {"job_key": job_key},
        {"$set": {
            "match_score": round(score, 4),
            "scored_at":   datetime.now(timezone.utc),
        }},
    )


def get_top_matches(collection, n=10):
    """Return the top-n jobs ranked by match_score (used in the reporting step)."""
    return list(
        collection.find({"match_score": {"$ne": None}}, {"_id": 0})
                  .sort("match_score", -1)
                  .limit(n)
    )


print("MongoDB helper functions defined.")

---
## Section 4 — Indeed Scraper

Defines the scraper function. Run this cell to load it — it does **not** scrape yet.

**How it works:**
1. Opens a headless (invisible) Chrome window using Playwright
2. Navigates to Indeed with your search query and location
3. Loops through each job card, clicks it to reveal the full description
4. Extracts: `title`, `company`, `location`, `salary`, `description`, `url`, `job_key`
5. Returns a list of job dicts ready to be inserted into MongoDB

**Anti-bot behavior built in:**
- Real Chrome user-agent string
- Random delays between card clicks (mimics human browsing speed)
- Errors on individual cards are caught and skipped so one bad card doesn't stop the whole batch

In [ ]:
import asyncio
import random
from datetime import datetime, timezone
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout

# A real browser user-agent so Indeed doesn't flag the request as a bot
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


def _build_search_url(query, location):
    """Build the Indeed search URL from a query string and location."""
    q = query.replace(" ", "+")
    l = location.replace(" ", "+").replace(",", "%2C")
    return f"https://www.indeed.com/jobs?q={q}&l={l}&sort=date"


async def _pause(min_s=1.5, max_s=3.5):
    """Wait a random number of seconds to mimic human browsing pace."""
    await asyncio.sleep(random.uniform(min_s, max_s))


async def scrape_indeed():
    """
    Main scraping coroutine. Launches a headless browser, searches Indeed,
    and returns a list of job dicts.
    """
    jobs = []
    url  = _build_search_url(CONFIG["SEARCH_QUERY"], CONFIG["SEARCH_LOCATION"])

    async with async_playwright() as pw:

        # Launch a headless Chromium browser (no visible window)
        browser = await pw.chromium.launch(headless=True)

        # Set a realistic browser context so we look like a normal user
        context = await browser.new_context(
            user_agent=USER_AGENT,
            viewport={"width": 1280, "height": 800},
            locale="en-US",
        )

        page = await context.new_page()

        # ----------------------------------------------------------------
        # Navigate to the Indeed search results page
        # ----------------------------------------------------------------
        print(f"Searching Indeed for '{CONFIG['SEARCH_QUERY']}' in '{CONFIG['SEARCH_LOCATION']}'")
        print(f"URL: {url}\n")

        await page.goto(url, wait_until="domcontentloaded", timeout=30_000)
        await _pause(2, 4)  # wait for JavaScript to finish rendering the cards

        # ----------------------------------------------------------------
        # Collect all job card elements on the page
        # Indeed gives each card a data-jk attribute with its unique job key
        # ----------------------------------------------------------------
        job_cards = await page.query_selector_all("[data-jk]")

        if not job_cards:
            print("No job cards found. Indeed may have updated its HTML or blocked the request.")
            await browser.close()
            return jobs

        # Limit to MAX_JOBS
        job_cards = job_cards[:CONFIG["MAX_JOBS"]]
        print(f"Found {len(job_cards)} job card(s). Extracting details...\n")

        # ----------------------------------------------------------------
        # Loop through each job card
        # ----------------------------------------------------------------
        for i, card in enumerate(job_cards, start=1):
            try:
                # Get the unique job key from the HTML attribute
                job_key = await card.get_attribute("data-jk")
                if not job_key:
                    continue

                # Extract surface-level fields directly from the card
                # data-testid selectors are more stable than class names
                title_el    = await card.query_selector('[data-testid="jobTitle"] span')
                company_el  = await card.query_selector('[data-testid="company-name"]')
                location_el = await card.query_selector('[data-testid="text-location"]')
                salary_el   = await card.query_selector('[data-testid="attribute_snippet_testid"]')

                title    = (await title_el.inner_text()).strip()    if title_el    else "N/A"
                company  = (await company_el.inner_text()).strip()  if company_el  else "N/A"
                location = (await location_el.inner_text()).strip() if location_el else "N/A"
                salary   = (await salary_el.inner_text()).strip()   if salary_el   else "N/A"

                print(f"  [{i}/{len(job_cards)}] {title} @ {company}")

                # Click the card to load the full description in the right panel
                await card.click()
                await _pause(1.5, 3.0)

                # Scrape the full description text from the job detail panel
                description = ""
                try:
                    desc_el = await page.wait_for_selector(
                        "#jobDescriptionText",
                        timeout=8_000,
                    )
                    description = (await desc_el.inner_text()).strip()
                except PlaywrightTimeout:
                    # Description panel didn't load in time — save the job anyway
                    print(f"    Warning: description timed out for {title}")

                # Build the MongoDB document for this job
                # match_score is None here — the NLP step fills it in later
                job_doc = {
                    "job_key":         job_key,
                    "title":           title,
                    "company":         company,
                    "location":        location,
                    "salary":          salary,
                    "description":     description,
                    "url":             f"https://www.indeed.com/viewjob?jk={job_key}",
                    "search_query":    CONFIG["SEARCH_QUERY"],
                    "search_location": CONFIG["SEARCH_LOCATION"],
                    "scraped_at":      datetime.now(timezone.utc),
                    "match_score":     None,
                }
                jobs.append(job_doc)

                await _pause(1.0, 2.5)  # polite pause between cards

            except Exception as e:
                # One bad card should never stop the whole batch
                print(f"    Error on card {i}: {e}")
                continue

        await browser.close()

    print(f"\nScraping complete — {len(jobs)} job(s) collected.")
    return jobs


print("Scraper function defined.")

---
## Section 5 — Run: Connect → Scrape → Store

This is the cell you run each week (or whenever you want fresh data).
It calls the functions defined above in order:

1. Connect to MongoDB Atlas
2. Scrape Indeed (takes 2–5 minutes depending on MAX_JOBS)
3. Insert results into MongoDB, skipping any duplicates

> **Note:** Jupyter notebooks don't support `asyncio.run()` directly because
> they already run an event loop internally. We use `await` directly instead,
> which only works inside a notebook cell.

In [ ]:
# ── Step 1: Connect to MongoDB ──────────────────────────────────────────────
print("=" * 55)
print("STEP 1: Connecting to MongoDB Atlas")
print("=" * 55)

db         = get_db()
collection = get_collection(db)
print()

# ── Step 2: Scrape Indeed ───────────────────────────────────────────────────
print("=" * 55)
print("STEP 2: Scraping Indeed")
print("=" * 55)

# `await` works directly in Jupyter because it already runs an async event loop
jobs = await scrape_indeed()
print()

# ── Step 3: Store in MongoDB ─────────────────────────────────────────────────
if not jobs:
    print("No jobs scraped — nothing to store. Check the scraper output above.")
else:
    print("=" * 55)
    print("STEP 3: Storing in MongoDB")
    print("=" * 55)

    summary = insert_jobs(collection, jobs)
    print()
    print(f"Done.  Inserted: {summary['inserted']}  |  Skipped (duplicates): {summary['skipped']}")

---
## Section 6 — Verify: Preview Stored Jobs

Run this cell anytime to confirm what's in MongoDB.
It prints the first 5 stored jobs and shows a summary count.

In [ ]:
all_jobs = get_all_jobs(collection)

print(f"Total jobs stored in MongoDB: {len(all_jobs)}")
print("=" * 55)

# Preview the first 5
for job in all_jobs[:5]:
    print(f"\nTitle       : {job.get('title')}")
    print(f"Company     : {job.get('company')}")
    print(f"Location    : {job.get('location')}")
    print(f"Salary      : {job.get('salary')}")
    print(f"Match score : {job.get('match_score')}  ← will be filled in by the NLP step")
    print(f"URL         : {job.get('url')}")
    desc_preview = (job.get('description') or '')[:120].replace('\n', ' ')
    print(f"Description : {desc_preview}...")

---
## Up Next — NLP Section (coming in the next step)

The following cells will be added here once the scraping and storage above is confirmed working:

| Step | What it does |
|---|---|
| **Section 7** | Upload your resume PDF and extract the text with PyMuPDF |
| **Section 8** | Embed the resume + all job descriptions using `sentence-transformers` |
| **Section 9** | Compute cosine similarity score for each job → write back to MongoDB |
| **Section 10** | Export a ranked report CSV for Power BI |